In [ ]:
import pandas as pd
import numpy as np

# Strategies for Working With Missing Data

You will often encounter datasets with incomplete data. For example, some cell:
- was not collected at all.
- came from a survey and the respondent refused to provide it.
- is not applicable for that row (e.g., in a survey, how many years the person owned their home if they don't own a home).
- came from an automated sensor and the sensor malfunctioned.
- is the result of running a script on other data, and that script wasn't run for this row.

Let's consider alternatives for representing missing data in Pandas, and the impacts of those alternatives on operations you might perform.

More details can be found in the [Pandas docs on handling missing data](https://pandas.pydata.org/pandas-docs/version/2.2/user_guide/missing_data.html).

## Representing missing data

There are a few possible different representations for missing data.

- A default value (Generally, this is a bad idea. You won't be able to distinguish a cell that truly has the default value from a cell where the data is missing.)
    - 0 (for numeric data)
    - '' (empty string)
    - False (for boolean)



- A special value
    - np.nan (aka NaN)
    - NaT (for datetimes)
    - None
    - NA (still experimental as of pandas 2.2)

In [ ]:
# np.nan is a good representation for numeric columns
df = pd.DataFrame({
    'A': [1, 2, np.nan],
    'B': [3, np.nan, 5],
    'C': [np.nan, 7, 8]
})
df

Note that the printed representation is NaN (not a number), but you have to refer to it in code as `np.nan`

All three columns are numeric, even though they contain missing values.

In [ ]:
df.dtypes

Adding two columns yields a result that makes sense. If either element is missing data, the sum is treated as missing data.

In [ ]:
df['A'] + df['B']

By the way, same kind of behavior happens with numpy arrays. (Not surprising, since pandas is actually built on top of numpy).

In [ ]:
# create a numpy array with the same contents as the DataFrame
arr = df.to_numpy()
arr

In [ ]:
# add the first two columns together
arr[:, 0] + arr[:, 1]

## Representing non-numeric missing data
For non-numeric missing data, np.nan isn't quite the right thing. So pandas has some other representations. Let's see.

In [ ]:
# Add a datetime column
df['D'] = pd.to_datetime(['2020-01-01', '2020-01-02', '2020-01-03'])
df

In [ ]:
df.dtypes

In [ ]:
# set the last cell to be missing
df.loc[2, 'D'] = np.nan
df

NaT (Not a Time) is the missing value for datetime columns. Pandas automatically converted np.nan to pd.NaT

In [ ]:
# Add a boolean column
df['Rainy'] = [True, False, True]
df

In [ ]:
df.dtypes

In [ ]:
# set the last cell to be missing
df.loc[2, 'Rainy'] = np.nan
df

In [ ]:
df.dtypes

In order to accomodate that np.nan value, it converted Rainy from being an all-boolean column to a more generic datatype, `object`

## pd.NA
Pandas uses pd.NA (printed representation <NA>) for missing Boolean values. It's still "experimental" as of pandas 2.2, meaning that in future versions of pandas they could change some things about how it works. But it's probably pretty stable at this point, because it's been in pandas since version 1.0

If I use the base python value None in a boolean column, it automatically gets converted to pd.NA

In [ ]:
# reset the Rainy column to be boolean
df['Rainy'] = df['Rainy'].astype('boolean')
df

In [ ]:
df.dtypes

In [ ]:
df.loc[2, 'Rainy'] = None
df

I can actually use pd.NA everywhere, even for floating point columns. It makes it easier, by providing a single representation of missing data for all datatypes. But keep in mind that it will get converted to one of the more specific null datatypes as needed.

In [ ]:
df.loc[0, 'A'] = pd.NA
df

In [ ]:
df.loc[1, 'D'] = pd.NA
df

In [ ]:
# alternatively, I can convert all the datatypes into more modern (extension) types that support pd.NA as the missing value
df = df.convert_dtypes()
df

In [ ]:
df.dtypes

## Identifying cells with missing values

Pandas provides a function,  `.isna()`, and a synonym for it, `.isnull`.

The reason for having both isna and isnull is largely historical: `.isnull()` is based on the original function name in R (a language that heavily influenced the development of pandas), while `.isna()` is more in line with naming conventions in python.






In [ ]:
# Running .isna() on a DataFrame returns a Boolean mask,
# a DataFrame of the same shape with True wherever the original DataFrame has missing data

df.isna()

In [ ]:
# same thing; isnull is a synonym for isna
df.isnull()

# Dropping Missing Data
Pandas has operations that will drop rows or columns that have missing data. It's not always necessary to do this, even when you are going to perform an operation like taking the sum of all values in a column, because many of the operations like taking the sum have special ways of handling null values, but that's a lesson for another time.

For now, it's worth understanding how to filter out missing data when you want to.

In [ ]:
# add a row without missing data
df.loc[3] = [4, 6, 8, pd.to_datetime('2020-01-04'), False]
# add a row with all missing data
df.loc[4] = [pd.NA, pd.NA, pd.NA, pd.NA, pd.NA]
df

In [ ]:
# drop rows where all data is missing
df.dropna(how='all')

In [ ]:
# drop rows where any data is missing
df.dropna(how='any')

In [ ]:
# default is 'any'
df.dropna()

In [ ]:
# note that dropna() returns a new DataFrame, it doesn't modify the original
df

As with many of these operations, if you specify `inplace=True` it will modify the original DataFrame

In [ ]:
new_df = df.copy()
#new_df.dropna(inplace=True)
new_df = new_df.dropna()
new_df

In [ ]:
df

## Filling Missing Data

To run some operations, you may want to convert missing values to some value you specify. You can do that in a few different ways.

In [ ]:
# use fillna() to replace missing data with a value
# working only on the numeric columns here
df[['A', 'B', 'C']].fillna(0)


In [ ]:
# or fill it with different values for each column
df.fillna({
    'A': 100,
    'B': 200,f
    'C': 300,
    'D': pd.to_datetime('2000-12-31'),
    'Rainy': False
})



In [ ]:
# or fill it with different values for each row
# again just including the three numeric columns
df[['A', 'B', 'C']].fillna({
    0: 0,
    1: 100,
    2: 200,
    3: 300,
    4: 400
}, axis=1)


But that's not implemented yet, even though it's should be...

Here's how I could do it, using the `apply` method

In [ ]:
#row.name is the value of the index for that row
df[['A', 'B', 'C']].apply(lambda row: row.fillna(row.name * 100), axis=1)